In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    coalesce,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_identification_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyIdentificationId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "PartyIdentificationKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "PartyIdentificationTypeCode string,"
        #     "PartyIdentificationType string,"
        #     "PartyIdentificationNumber string,"
        #     "SourceTable string,"
        #     "MainFlag string,"
        #     "IssueAuthority string,"
        #     "IssueDate date,"
        #     "ExpiredDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PartyIdentification"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== eaiSIG_BP ==================

        eaiSIG_BP_extra_filters = None

        if eaiSIG_BP_extra_filters is not None:
            eaiSIG_BP_extra_filters.append(date_filter)
            eaiSIG_BP_final_filters = " AND ".join(eaiSIG_BP_extra_filters)
        else:
            eaiSIG_BP_final_filters = date_filter

        eaiSIG_BP_selected_fields = [
            "eaiSIG_BP_KEY",
            "eaiSIG_BP_CUST_IDNUM",
            "eaisig_bp_cust_s_unid",
            "businessdate",
        ]

        eaiSIG_BP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_BP",
            eaiSIG_BP_final_filters,
            eaiSIG_BP_selected_fields,
        )

        # =============== Join with portfolio ===============
        eaiSIG_BP = eaiSIG_BP.join(
            portfolio,
            on=substring(eaiSIG_BP["eaisig_bp_cust_s_unid"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        # ================== Join ==================
        PartyIdentificationId_primary_key_fields = [
            eaiSIG_BP.eaiSIG_BP_KEY,
            eaiSIG_BP.eaiSIG_BP_CUST_IDNUM,
            lit("arotron_eaiSIG_BP"),
            lit("arotron"),
        ]

        PartyIdentificationKey_primary_key_fields = [
            eaiSIG_BP.eaiSIG_BP_KEY,
            eaiSIG_BP.eaiSIG_BP_CUST_IDNUM,
            lit("arotron_eaiSIG_BP"),
        ]

        # ================== party_identification_arotron ==================
        party_identification_arotron = (
            eaiSIG_BP.withColumn(
                "PartyIdentificationId",
                sha2(concat_ws(";", *PartyIdentificationId_primary_key_fields), 256),
            )
            .withColumn(
                "PartyIdentificationKey",
                concat_ws(";", *PartyIdentificationKey_primary_key_fields),
            )
            .where(
                eaiSIG_BP.eaiSIG_BP_KEY.isNotNull()
                & eaiSIG_BP.eaiSIG_BP_CUST_IDNUM.isNotNull()
                & (eaiSIG_BP.eaiSIG_BP_CUST_IDNUM != "")
            )
        )

        party_identification_arotron = party_identification_arotron.select(
            party_identification_arotron.PartyIdentificationId.cast("string").alias(
                "PartyIdentificationId"
            ),
            lit("arotron").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            party_identification_arotron.PartyIdentificationKey.cast("string").alias(
                "PartyIdentificationKey"
            ),
            party_identification_arotron.eaiSIG_BP_KEY.cast("string").alias("PartyKey"),
            party_identification_arotron.eaisig_bp_cust_s_unid.cast("string").alias(
                "CDI"
            ),
            lit("IdentityCard").alias("PartyIdentificationTypeCode"),
            lit("IdentityCard").alias("PartyIdentificationType"),
            party_identification_arotron.eaiSIG_BP_CUST_IDNUM.cast("string").alias(
                "PartyIdentificationNumber"
            ),
            lit("arotron_eaiSIG_BP").cast("string").alias("SourceTable"),
            lit(None).cast("string").alias("MainFlag"),
            lit(None).cast("string").alias("IssueAuthority"),
            lit(None).alias("IssueDate"),
            lit(None).alias("ExpiredDate"),
            party_identification_arotron.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        party_identification_arotron.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_party_identification_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)